# Lab 7 — Large Language Models (LLMs)

**Name:** Omar Hejazi
**Student ID:** 2230009001

This notebook works through the three transformer families using small open models that run on
CPU, then answers Tasks 1, 2, 3 and 5 and the optional challenge. The lab sheet skips Task 4;
the numbering below follows the sheet rather than renumbering it.

Every number and model output quoted in the written sections comes from the executed cells in
this notebook. Where a model produced a poor answer I have kept the poor answer and explained it
instead of swapping in a better prompt, because the failures are the most informative part of
working with models this small.

## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |

The split comes down to which tokens each position is allowed to attend to. An encoder uses
bidirectional attention, so every token sees the whole sentence — good for building a
representation, useless for generating text left to right. A decoder uses causal (masked)
attention, so a token only sees what came before it, which is exactly what next-token prediction
needs. An encoder–decoder keeps both: a bidirectional read of the input, and a causal decoder
that writes the output while attending back to that read.

## 1. Setup

In [1]:
!pip -q install transformers sentencepiece torch


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings

warnings.filterwarnings("ignore")

from transformers import pipeline, set_seed, logging as hf_logging

# The model-loading and generation warnings are informational and make the
# outputs below hard to read; the deprecation notices do not apply to anything
# this notebook does.
hf_logging.set_verbosity_error()

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
import transformers, torch

print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

transformers: 5.19.0
torch       : 2.6.0+cu124
CUDA available: True


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.

In [4]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 20313.99it/s]

Natural language processing is a common feature of the language processing process.























`do_sample=False` is greedy decoding: at every step the model takes the single highest-probability
token. The output above shows what that costs. After one short clause the continuation collapses
into a run of blank lines and stops saying anything.

This is not a bug. Greedy decoding is deterministic, so once the model reaches a state where the
most likely next token is a newline, the state after emitting that newline is almost the same, so
the most likely token is a newline again. There is nothing in the procedure to break the cycle.
The repetition is a property of taking the argmax at every step, not evidence that the model
assigns high probability to an infinite run of newlines.

That diagnosis suggests two different ways out, and they are worth separating because they cost
different things. The loop can be broken by making the choice random — sampling, which the
optional challenge at the end measures — or by forbidding the repetition directly, which keeps
the decoding deterministic. The next cell tries the second option, since giving up
reproducibility would be a steep price to pay for one fix.

In [5]:
# Two decoding constraints that attack the repetition directly, with do_sample=False
# throughout, so every row here is still deterministic.
#
#   no_repeat_ngram_size=3  bans any 3-gram that has already appeared
#   repetition_penalty=1.2  divides the logit of any already-used token
decoding_variants = {
    "greedy (baseline)":           dict(do_sample=False),
    "greedy + no_repeat_ngram=3":  dict(do_sample=False, no_repeat_ngram_size=3),
    "greedy + rep_penalty=1.2":    dict(do_sample=False, repetition_penalty=1.2),
    "greedy + both":               dict(do_sample=False, no_repeat_ngram_size=3,
                                        repetition_penalty=1.2),
    "beams=4 + no_repeat_ngram=3": dict(do_sample=False, num_beams=4, no_repeat_ngram_size=3),
}

for name, kwargs in decoding_variants.items():
    runs = [generator(prompt, max_new_tokens=30, **kwargs)[0]["generated_text"]
            for _ in range(2)]
    print(f"{name}")
    print(f"    {runs[0]!r}")
    print(f"    identical on a second run: {runs[0] == runs[1]}\n")

greedy (baseline)
    'Natural language processing is a common feature of the language processing process.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'
    identical on a second run: True



greedy + no_repeat_ngram=3
    "Natural language processing is a common feature of the language processing process.\n\n\nThe language processing system is designed to process information from the user's computer, and to process"
    identical on a second run: True



greedy + rep_penalty=1.2
    'Natural language processing is a common feature of the Internet.\nThe term "language processing" has been used to describe many languages, but it does not mean that there are'
    identical on a second run: True



greedy + both
    'Natural language processing is a common feature of the Internet.\nThe term "language processing" has been used to describe many languages, but it does not mean that there are'
    identical on a second run: True



beams=4 + no_repeat_ngram=3
    'Natural language processing is the most common language processing technique in the world.\n\n\nThe following is a list of the most commonly used language processing techniques.\n1.'
    identical on a second run: True



Every variant above reports `identical on a second run: True`, including the baseline. That is the
first thing to take from the table: none of these knobs introduces randomness, so the collapse was
never a symptom of determinism as such. It was a symptom of *unconstrained* determinism.

`no_repeat_ngram_size=3` bans any three-token sequence that has already been emitted. It partly
works — the output escapes into a real second sentence — but three newlines still survive, because
the ban only triggers once `\n\n\n` has appeared a first time. A hard constraint on repeats
removes the loop without removing what caused it.

`repetition_penalty=1.2` works better, and for a different reason. Rather than forbidding a
repeat outright it down-weights the logit of every token already used, so the newline becomes
progressively less attractive each time it is chosen and the argmax moves elsewhere on its own.
The result is the cleanest continuation in the table and it arrives with no blank-line run at all.

Combining both is indistinguishable from the penalty alone here: the penalty already keeps the
text clear of any repeated 3-gram, so the n-gram ban never fires and has nothing to do. Beam
search with the same ban also escapes the loop, though it drifts into list-making.

So the first finding has a fix that keeps everything useful about greedy decoding. The honest
caveat is that the content is still unreliable — "a common feature of the Internet" is not a
correct description of NLP. Fixing degenerate repetition makes the output readable, not true, and
those are separate problems with separate causes.

In [6]:
# The same prompt and the same model, decoded by sampling instead of argmax.
set_seed(42)
sampled = generator(prompt, max_new_tokens=30, do_sample=True, top_k=50)
print(sampled[0]["generated_text"])

Natural language processing is a common technique that has been used extensively in a variety of languages, including English, French, German, Spanish, Swedish, and Dutch. It can


With sampling the model produces a fluent, grammatical sentence. It is also wrong — the list of
languages it offers is not a claim about NLP that any source supports; it is a plausible-sounding
continuation. That distinction matters for the rest of the lab: a decoder-only model optimises
for likely text, and likely text is not the same thing as true text.

# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"))

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 27873.03it/s]

Je s'agissant de la technologie de linguage naturelle.


The model clearly understood the instruction — it switched languages, which means the
`Translate to French:` prefix was read as a task and not as text to continue. The French itself
is not correct. `Je s'agissant` is not a grammatical construction, and `linguage naturelle`
mixes a non-word with the wrong adjective agreement.

So the instruction-following behaviour and the translation quality come apart. FLAN-T5-small is
about 80M parameters and was instruction-tuned on a mixture that is overwhelmingly English; it
has learned the shape of the task far better than it has learned French morphology. A larger
checkpoint in the same family (`flan-t5-base` or `-large`) is the fix, not a better prompt.

In [8]:
# A convenience wrapper, so the rest of the notebook is not repeating the
# tokenize / generate / decode sandwich at every call.
def ask_t5(prompt, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(**inputs, **generate_kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)


print(ask_t5("Translate to French: I love natural language processing."))

Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.

In [9]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 22650.72it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


POSITIVE with a score of 0.9998. The score is a softmax probability over exactly two classes,
POSITIVE and NEGATIVE, because the classification head was fine-tuned on SST-2, which has two
labels. There is no neutral option available to this model at all — a point that turns out to
matter in Task 2.

# Tasks

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

| # | Task | Family | Why |
|---|---|---|---|
| 1 | Sentiment classification | **Encoder-only** | One label for the whole input. Bidirectional attention lets every token see the negations and contrastive cues around it, and no text has to be generated. |
| 2 | Machine translation | **Encoder–decoder** | Input and output are both sequences, in different languages, and of different lengths. The encoder reads the source once; the decoder emits the target while attending back to it. |
| 3 | Open-ended text generation | **Decoder-only** | There is no separate input to encode — the prompt and the continuation are one stream. Causal attention and next-token prediction are exactly this task. |
| 4 | Named Entity Recognition | **Encoder-only** | A label per token, not per sentence, so we want one contextual vector per token and a classifier on top. Deciding that "Washington" is a person rather than a place needs the tokens on both sides. |
| 5 | Text summarization | **Encoder–decoder** | Long input, short generated output, output grounded in the input. This is the task BART and T5 were built around. |

Two of these are conventions rather than hard limits, and it is worth being precise about which.

Translation and summarization (2 and 5) are routinely done today by large decoder-only models
that simply read the instruction and the source text as part of the prompt — that is what happens
when ChatGPT is asked to translate something. The encoder–decoder answer is the right one for
this lab because it is the architecture *designed* for the input-to-output mapping, and at the
scale we can run on CPU it is also the one that actually works.

Tasks 1 and 4, by contrast, are genuine architectural preferences. A decoder-only model can be
made to emit a label, but its causal mask means the representation of each token has not seen the
words that follow it, which is a real handicap for per-token tagging.

## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [10]:
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

for sentence in sentences:
    prediction = classifier(sentence)[0]
    print(f"{prediction['label']:8s}  {prediction['score']:.4f}   {sentence}")

POSITIVE  0.9999   The movie was excellent.
NEGATIVE  0.9997   The lecture was difficult to understand.
POSITIVE  0.9997   I enjoyed learning about transformers.


All three predictions are the expected ones, and all three are returned with a confidence above
0.999.

That uniformity is itself the thing to question. These are short, lexically unambiguous sentences
built around a single strong cue word — *excellent*, *difficult*, *enjoyed* — which is close to
the easiest case this model will ever see. A score of 0.9997 says the model is far from the
decision boundary on this input; it says nothing about how well calibrated the model is in
general. The next cell tries inputs that are harder on purpose.

In [11]:
# Three cases chosen to probe specific weaknesses rather than to confirm the result above.
probe_sentences = [
    "The exam was hard but the teaching was outstanding.",  # contrast: two opposite cues
    "I wouldn't say I disliked it.",                        # double negation
    "The train arrives at 6pm.",                            # no sentiment at all
]

for sentence in probe_sentences:
    prediction = classifier(sentence)[0]
    print(f"{prediction['label']:8s}  {prediction['score']:.4f}   {sentence}")

POSITIVE  0.9997   The exam was hard but the teaching was outstanding.
POSITIVE  0.7383   I wouldn't say I disliked it.
POSITIVE  0.9681   The train arrives at 6pm.


Three distinct behaviours here.

**The contrastive sentence is handled correctly.** "The exam was hard but the teaching was
outstanding" comes back POSITIVE at 0.9997. A bag-of-words model would struggle, since *hard* and
*outstanding* simply both appear; getting this right requires noticing that *but* subordinates the
first clause. Bidirectional self-attention is what makes that available, and this is the clearest
single piece of evidence in the notebook for why an encoder is the right family for Task 1.

**Double negation is the weakest case.** "I wouldn't say I disliked it" returns POSITIVE at 0.738
— the only score in this notebook anywhere near the decision boundary. The label is defensible,
and so is the low confidence: the sentence is a grudging concession, and stacking *wouldn't* on
*disliked* is genuinely hard. Here the confidence score is doing useful work.

**The neutral sentence is the actual failure.** "The train arrives at 6pm." carries no sentiment,
and the model reports POSITIVE at 0.968. This is not the model being wrong so much as being asked
an impossible question. The SST-2 head has two output units; the softmax over them must sum to 1,
so no input can ever be assigned "neither". A high score for POSITIVE here means only *more
positive than negative*, which on a neutral input is noise amplified to near-certainty.

The practical consequence: a confidence threshold cannot be used to filter out neutral text with
this model, because neutral text does not produce low confidence. Detecting neutrality needs a
head trained with a neutral class, not a cleverer use of these two scores.

## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.

In [12]:
paragraph = (
    "Transformer models process every token in a sentence at the same time instead of "
    "reading it word by word. Each token is compared against every other token through "
    "an attention mechanism, so the model can connect words that sit far apart. This made "
    "training much faster on modern hardware and is the reason transformers replaced "
    "recurrent networks in most natural language processing systems."
)

print("input length:", len(tokenizer(paragraph)["input_ids"]), "tokens")
print()
print("summary:", ask_t5("Summarize: " + paragraph, max_new_tokens=60))

input length: 73 tokens



summary: Read the tokens in a sentence.


The summary is `Read the tokens in a sentence.`

This is a poor result in three separate ways, and they have different causes. It is phrased as an
instruction rather than a statement; it drops the attention mechanism, the speed gain and the
comparison with recurrent networks; and the one idea it keeps, reading tokens, it states in a form
that loses the point the paragraph was actually making — that tokens are processed *together*
rather than one at a time.

Note also that the output is far shorter than the 60 tokens allowed, so length is not the
constraint. The model emitted its end-of-sequence token early because it had nothing more it
wanted to say. The next two cells test that explanation rather than assuming it.

In [13]:
# If the model stopped early for lack of anything to add, forcing it to keep
# going should produce padding rather than more content.
print("min_new_tokens=25:")
print(ask_t5("Summarize: " + paragraph, max_new_tokens=60, min_new_tokens=25))
print()
print("num_beams=4:")
print(ask_t5("Summarize: " + paragraph, max_new_tokens=60, num_beams=4))

min_new_tokens=25:


Read the tokens in a sentence. Read the tokens in a sentence. Read the tokens in a sentence. Read the tokens in a sentence. Read the tokens in a sentence. Read the tokens in a sentence. Read the tokens in 

num_beams=4:


Learn how transformers work. Learn how transformers work. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers. Learn transformers.


Both decoding changes confirm it. Blocking the end-of-sequence token with `min_new_tokens=25`
makes the model repeat "Read the tokens in a sentence." six times — it fills the quota with
copies instead of adding information. Beam search, which is usually the right choice for
summarization, degenerates the same way into "Learn transformers." repeated a dozen times.

So the short summary is not a decoding problem and cannot be fixed by tuning generation
parameters. The model's own distribution over summaries of this paragraph is thin.

In [14]:
# Prompt sensitivity: the same paragraph, the same model, four phrasings of the request.
prompt_templates = {
    "Summarize: <text>":
        "Summarize: " + paragraph,
    "Summarize ... in one sentence:\n<text>":
        "Summarize the following paragraph in one sentence:\n" + paragraph,
    "<text>\nSummarize the paragraph above.":
        paragraph + "\nSummarize the paragraph above.",
    "Write a one-sentence summary ...\n<text>":
        "Write a one-sentence summary of this text.\n" + paragraph,
}

for label, prompt_text in prompt_templates.items():
    print(f"{label}\n    -> {ask_t5(prompt_text, max_new_tokens=60)}\n")

Summarize: <text>
    -> Read the tokens in a sentence.



Summarize ... in one sentence:
<text>
    -> Read the tokens in a sentence at the same time.



<text>
Summarize the paragraph above.
    -> Transformers are a way to learn more about the language.



Write a one-sentence summary ...
<text>
    -> The transformers are able to learn the word by word.



Four phrasings of one request, four different summaries:

| Prompt | Output |
|---|---|
| `Summarize: <text>` | Read the tokens in a sentence. |
| `Summarize ... in one sentence:` before the text | Read the tokens in a sentence at the same time. |
| Instruction placed *after* the text | Transformers are a way to learn more about the language. |
| `Write a one-sentence summary ...` | The transformers are able to learn the word by word. |

Two things stand out.

Moving the instruction after the paragraph changes the output the most. That is a clue about the
encoder: it reads the whole input bidirectionally in one pass, so the instruction is not
"remembered" from earlier the way it would be in a causal decoder — it is part of a single
representation, and its position changes that representation.

The best-sounding output is the most wrong. "The transformers are able to learn the word by word"
is fluent, on-topic, and asserts the exact opposite of the source paragraph, whose first sentence
says transformers process tokens simultaneously **instead of** word by word. The second output,
which keeps "at the same time", is clumsier English but factually closer. Fluency and faithfulness
are independent, and picking the summary that reads best would have picked the one that inverts
the source.

A reasonable conclusion for a model this size: it is adequate for extracting a short answer that
is present verbatim in the text, and not adequate for abstractive summarization. The next cell
checks the first half of that claim.

In [15]:
# Extractive question answering on the same paragraph, as a contrast with summarization.
question = "What mechanism do transformers use to connect distant words?"
print(ask_t5(f"Answer the question. {question} Context: {paragraph}", max_new_tokens=20))

attention


`attention` — correct, and from the same model and the same paragraph that produced the three bad
summaries above.

This locates the limitation fairly precisely. Reading the paragraph is not the problem: the
encoder built a representation good enough to find the right span. What the model cannot do is
compress five clauses into one new sentence, which requires deciding what matters and then
composing text that was never in the input. Understanding is cheaper than generation here.

## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**

A conversation is just one growing stream of text — system prompt, user turns and assistant turns
concatenated — and a decoder-only model's causal attention means each new token is predicted from
everything already in that stream, which is exactly the shape of the problem. Because there is no
separate encoder, the question does not have to be separated from the answer or from the twenty
turns before it: context and continuation are the same sequence, so the model handles follow-ups
and references to earlier messages without any architectural special-casing. Training is also
cheap in the sense that matters, since next-token prediction over ordinary text supervises every
position at once and needs no paired input/output data, which is what makes pre-training on
internet-scale corpora possible. That scale is what produced the general capability: as Part B
showed, a small encoder–decoder follows the *form* of an instruction while getting the content
wrong, and the fix was a bigger model rather than a different prompt. The trade-off is visible in
Part A — a decoder-only model optimises for likely text, so it produced a fluent and completely
unfounded sentence, which is why chat models need alignment training on top of next-token
prediction rather than next-token prediction alone.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.

In [16]:
# Part B reassigned `prompt` to the translation request, so the generation
# prompt is restated explicitly here rather than relied on from Part A.
gen_prompt = "Natural language processing is"

# Case 1: sampling with the seed reset immediately before each call.
for run in range(1, 3):
    set_seed(42)
    text = generator(gen_prompt, max_new_tokens=30, do_sample=True, top_k=50)[0]["generated_text"]
    print(f"run {run}: {text}\n")

run 1: Natural language processing is a common technique that has been used extensively in a variety of languages, including English, French, German, Spanish, Swedish, and Dutch. It can



run 2: Natural language processing is a common technique that has been used extensively in a variety of languages, including English, French, German, Spanish, Swedish, and Dutch. It can



In [17]:
# Case 2: sampling, seeded once, without resetting between calls.
set_seed(7)
for run in range(1, 4):
    text = generator(gen_prompt, max_new_tokens=30, do_sample=True, top_k=50)[0]["generated_text"]
    print(f"run {run}: {text}\n")

run 1: Natural language processing is not always the best way to understand some of the most important problems you are experiencing.



These problems are usually caused by a single process



run 2: Natural language processing is a powerful and effective tool to help you improve your learning and learning.





The ability to understand the language is one of the



run 3: Natural language processing is a great way to help you learn and to learn.






















In [18]:
# Case 3: greedy decoding, no seed involved at all.
for run in range(1, 3):
    text = generator(gen_prompt, max_new_tokens=30, do_sample=False)[0]["generated_text"]
    print(f"run {run}: {text!r}\n")

run 1: 'Natural language processing is a common feature of the language processing process.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'



run 2: 'Natural language processing is a common feature of the language processing process.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'



### Why the outputs differ

The three cases above separate two things that are easy to conflate.

**Case 1** reseeds before every call and produces the *same* sentence twice, character for
character. **Case 2** seeds once and then samples three times, and gets three different
sentences. **Case 3** uses greedy decoding with no seed at all and is identical on both runs.

So the variation does not come from the model. The weights are fixed, and for a given input the
forward pass returns the same probability distribution over the next token every time. What
changes is how a token is *chosen* from that distribution. Greedy decoding takes the argmax, which
is a deterministic function of the distribution — hence Case 3. Sampling draws from the
distribution using the global random number generator, and each draw advances that generator's
state, so the second call in Case 2 starts from a different state than the first and goes down a
different path. `set_seed(42)` resets that state, which is why Case 1 repeats exactly.

`top_k=50` restricts each draw to the 50 most likely tokens, which bounds how strange the output
can get but does not make it deterministic — it is still a random draw, just from a shorter list.

The effect compounds. One different token early on changes the input to every later step, so the
continuations diverge completely rather than differing in a word or two — compare the three
sentences in Case 2, all grown from the same four-word prompt.

This is also the practical reason the Case 3 output is unusable. Greedy decoding is reproducible
and collapses into repeated newlines; sampling is fluent and not reproducible. The two properties
trade off directly, and when a reproducible run of a sampling model is needed, the seed is what
buys it back.

# Summary

**Results**

- DistilGPT-2, greedy: the continuation collapses into repeated newlines after one clause.
  `repetition_penalty=1.2` removes the collapse while staying fully deterministic, and
  `no_repeat_ngram_size=3` only partly does. With `do_sample=True, top_k=50` the same model and
  prompt produce a fluent sentence whose content is unsupported.
- FLAN-T5-small follows the `Translate to French:` instruction — it does switch language — but
  returns ungrammatical French (`Je s'agissant de la technologie de linguage naturelle`).
- DistilBERT/SST-2 labels all three Task 2 sentences correctly at above 0.999 confidence, and also
  handles a contrastive sentence correctly. It assigns POSITIVE 0.968 to the sentimentless
  "The train arrives at 6pm."
- FLAN-T5-small summarization is weak in a way that decoding parameters do not fix: `min_new_tokens`
  and beam search both produce repetition rather than more content. The same model answers an
  extractive question about the same paragraph correctly (`attention`).
- Sampling with the seed reset before each call reproduces exactly; seeded once and called
  repeatedly it does not; greedy decoding is deterministic without any seed.

**What the three families are actually for**

Encoder-only wins where the whole input must be seen at once and nothing needs generating, and the
contrastive-sentence result is the evidence. Encoder–decoder wins where output length differs from
input length and the output must stay grounded in the input. Decoder-only wins where the prompt and
the continuation are one stream, which is why it is the right shape for a chatbot.

**The limitation worth carrying forward**

At this scale, understanding is consistently cheaper than generation. FLAN-T5-small could locate
the right span in the paragraph but could not compose a faithful one-sentence summary of it, and
the most fluent summary it produced asserted the opposite of the source. Judging these models by
how well their output reads would have ranked the worst answer first.